In [38]:
import numpy as np
import trimesh
import matplotlib.pyplot as plt

mesh_path = "/Users/chaitanyaattanti/Downloads/MIXAR/8samples/cylinder.obj"

# Load original mesh with process=False to prevent vertex merging
mesh = trimesh.load(mesh_path, process=False)

orig_vertices = mesh.vertices.copy()
faces = mesh.faces.copy()

BINS = 1024

print("Loaded original mesh:", orig_vertices.shape)


Loaded original mesh: (192, 3)


In [39]:
def load_ply_vertices(path):
    """
    Safe loader for .ply files using trimesh with process=False
    Ensures vertex order & count remain unchanged.
    """
    m = trimesh.load(path, process=False)
    return m.vertices.copy()


In [ ]:
# Load normalized + quantized files safely
minmax_norm = load_ply_vertices("/Users/chaitanyaattanti/Downloads/MIXAR/Min_Max_norm.ply")
minmax_quant = load_ply_vertices("/Users/chaitanyaattanti/Downloads/MIXAR/Min_Max_quantized.ply")

unit_norm = load_ply_vertices("/Users/chaitanyaattanti/Downloads/MIXAR/Unit_sphere_normalized.ply")
unit_quant = load_ply_vertices("/Users/chaitanyaattanti/Downloads/MIXAR/unit_sphere_quantized.ply")


print("Original shape:", orig_vertices.shape)
print("Min–Max norm shape:", minmax_norm.shape)
print("Min–Max quant shape:", minmax_quant.shape)
print("Unit norm shape :", unit_norm.shape)
print("Unit quant shape:", unit_quant.shape)

Original shape: (192, 3)
Min–Max norm shape: (192, 3)
Min–Max quant shape: (192, 3)
Unit norm shape : (192, 3)
Unit quant shape: (192, 3)


In [ ]:
# 3D plot helper
def plot_vertices(points, title, filename):
    fig = plt.figure(figsize=(7,6))
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(points[:,0], points[:,1], points[:,2], s=1)
    ax.set_title(title)
    fig.savefig(filename, dpi=300)
    plt.close()

# Error calculation
def compute_error(original, reconstructed):
    mse = np.mean((original - reconstructed)**2, axis=0)
    mae = np.mean(np.abs(original - reconstructed), axis=0)
    return mse, mae



# Min–Max Reconstruction

In [ ]:
print("\nReconstructing Min–Max Normalization...")

vt_min = orig_vertices.min(axis=0)
vt_max = orig_vertices.max(axis=0)

# Dequantize
minmax_dequant = minmax_quant / (BINS - 1)

# Denormalize
minmax_reconstructed = minmax_dequant * (vt_max - vt_min) + vt_min

pc = trimesh.points.PointCloud(minmax_reconstructed)
pc.export("Min_Max_reconstructed.ply")

plot_vertices(minmax_reconstructed,
              "Min–Max Reconstructed",
              "Min_Max_reconstructed.png")

minmax_mse, minmax_mae = compute_error(orig_vertices, minmax_reconstructed)
print("Min–Max MSE:", minmax_mse)
print("Min–Max MAE:", minmax_mae)



Reconstructing Min–Max Normalization...
Min–Max MSE: [1.19493876e-06 0.00000000e+00 1.19493876e-06]
Min–Max MAE: [0.00091642 0.         0.00091642]


Unit Sphere Reconstruction

In [46]:
print("\nReconstructing Unit Sphere Normalization...")

center = orig_vertices.mean(axis=0)
radius = np.max(np.linalg.norm(orig_vertices - center, axis=1))

# Dequantize
unit_dequant = unit_quant / (BINS - 1)

# Shift [0,1] → [-1,1]
unit_shifted = (unit_dequant * 2) - 1

# Denormalize
unit_reconstructed = (unit_shifted * radius) + center

# ------- FIXED EXPORT (No binary flag needed) -------
pc = trimesh.points.PointCloud(unit_reconstructed)
pc.export("Unit_sphere_reconstructed.ply")

# ----------------------------------------------------

plot_vertices(unit_reconstructed,
              "Unit Sphere Reconstructed",
              "Unit_sphere_reconstructed.png")

unit_mse, unit_mae = compute_error(orig_vertices, unit_reconstructed)
print("Unit Sphere MSE:", unit_mse)
print("Unit Sphere MAE:", unit_mae)



Reconstructing Unit Sphere Normalization...
Unit Sphere MSE: [2.52570235e-06 2.66928610e-06 2.52570235e-06]
Unit Sphere MAE: [0.00138242 0.00138242 0.00138242]


MSE Error Plot

In [47]:
axes = ["X", "Y", "Z"]
x = np.arange(3)

plt.figure(figsize=(8,5))
plt.bar(x - 0.15, minmax_mse, width=0.3, label="Min-Max")
plt.bar(x + 0.15, unit_mse, width=0.3, label="Unit Sphere")

plt.xticks(x, axes)
plt.ylabel("MSE")
plt.title("Reconstruction Error per Axis")
plt.legend()
plt.savefig("Reconstruction_Error_MSE.png", dpi=300)
plt.close()

print("\nTask 3 Completed Successfully!")



Task 3 Completed Successfully!
